# Day 59: Time Series Decomposition

## Install Note
`scikit-learn` is built for tabular, independent-sample data and doesn't have dedicated time series tools — it has no built-in notion of "this row comes chronologically after that one." `statsmodels` fills that gap with statistical tools purpose-built for time series: decomposition, ARIMA, and related models we'll use over the next few days. If `statsmodels` isn't already installed in your environment, run the pip command below once (in a terminal, or by uncommenting it here); this is just a setup note, not something the rest of the notebook depends on running again.

In [ ]:
# pip install statsmodels

## Creating a Seasonal Series
Almost every real-world time series (sales, temperature, website traffic) can be thought of as a combination of three things: a long-term trend (the series generally rising or falling over time), seasonality (a pattern that repeats at a fixed period, like higher sales every December), and noise/residual (random fluctuation that isn't explained by trend or seasonality). Understanding this before modeling matters because different forecasting techniques handle these components differently, and mixing them up (e.g. mistaking noise for a real pattern) leads to bad forecasts. This cell manually constructs a synthetic monthly series with all three ingredients so we can decompose it and check that we recover what we put in: a rising `trend` (linear from 100 to 160), a repeating `seasonality` (a sine wave completing 4 full cycles over 48 months, i.e. a 12-month period), and random Gaussian `noise` — then adds them together into one `series`.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(0)
# 48 monthly timestamps starting Jan 2021 (4 years of monthly data)
dates = pd.date_range('2021-01-01', periods=48, freq='M')
# Long-term upward trend: rises steadily from 100 to 160 over the 48 months
trend = np.linspace(100, 160, 48)
# Repeating seasonal pattern: a sine wave that completes a full cycle every 12 months
seasonality = 10 * np.sin(np.linspace(0, 8 * np.pi, 48))
# Random noise so the series isn't perfectly clean, like real-world measurement variation
noise = np.random.normal(0, 3, 48)
# Additive model: observed value = trend + seasonality + noise
series = pd.Series(trend + seasonality + noise, index=dates)

## Decomposing the Series
`seasonal_decompose` reverses what we did above: given a single observed series, it statistically separates it back into trend, seasonal, and residual components. `model='additive'` assumes the series is trend + seasonal + residual (appropriate when seasonal swings stay roughly constant in absolute size regardless of the trend level — use `model='multiplicative'` instead when seasonal swings grow proportionally as the trend grows, e.g. percentage-based fluctuations). `period=12` tells it the seasonal cycle repeats every 12 observations, which is correct here since we generated monthly data with a 12-month cycle — getting this period right is essential, since a wrong period will make the decomposition find nonsensical "seasonal" patterns. The resulting plot stacks four panels: the original series, the extracted trend, the extracted seasonal pattern, and the leftover residual (noise) — visually confirming whether the decomposition recovered structure that matches what you'd expect from the data.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# additive model: observed = trend + seasonal + residual; period=12 matches our monthly seasonality
result = seasonal_decompose(series, model='additive', period=12)
result.plot()  # plots original, trend, seasonal, and residual components stacked
import matplotlib.pyplot as plt
plt.show()

## Why Decomposition Matters
Decomposition isn't just a diagnostic plot — it directly informs modeling choices later on. If you can see the trend and seasonal components clearly, you know your forecasting model needs to account for both (e.g. picking a seasonal ARIMA order, or feeding seasonal indicators into an ML model), rather than naively fitting a model that assumes the series is stationary. It also helps you separate genuine signal from noise: the residual component is what's left after removing trend and seasonality, and if it looks like unpredictable random fluctuation (no visible pattern), that's a sign the model captured the real structure well. Practically, teams also use decomposition to deseasonalize a series (subtract out the seasonal component) so they can compare periods fairly, e.g. is this month's sales actually up, or is it just the usual December bump?

In [ ]:
# trend has NaNs at the start/end (not enough surrounding points to compute a moving average there),
# so dropna() shows only the valid computed values
print('Trend (first 5):', result.trend.dropna().head())
print('Seasonal (first 5):', result.seasonal.head())